# PathMNIST: FedAvg vs FedProx Under Non-IID Label Noise
### Complete GPU Execution Pipeline (Google Colab)

This notebook executes the complete experimental matrix on a GPU (T4 / A100 / V100):
1. **Pre-experimental $\mu$ sweep** on noisy seed 42 ($\mu \in \{0.001, 0.01, 0.1\}$)
2. **Automatic $\mu$ selection** using mean validation accuracy over rounds 91–100
3. **Direct reuse** of the winning sweep run as `fedprox_noisy_seed42`
4. **Full 2×2 factorial matrix** across 3 independent seeds (42, 123, 2026):
   - FedAvg Clean (Seeds 42, 123, 2026)
   - FedAvg Noisy (Seeds 42, 123, 2026)
   - FedProx Clean (Seeds 42, 123, 2026)
   - FedProx Noisy (Seeds 42, 123, 2026)
5. **Compilation of deliverables**: full numerical tables, diagnostic plots, and an executive 1-page PDF report (`PathMNIST_FedAvg_vs_FedProx_Report.pdf`).

## Step 1: Mount Google Drive by Default (Persistent Storage)
Google Drive is mounted by default so all checkpoints, logs, and deliverables persist across disconnects.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
drive_results_dir = '/content/drive/MyDrive/PathMNIST_Results'
os.makedirs(drive_results_dir, exist_ok=True)
print(f'Persistent results directory configured: {drive_results_dir}')

## Step 2: Verify GPU Acceleration
Ensure your Colab runtime is set to GPU: **Runtime -> Change runtime type -> T4 GPU (or A100)**.

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), 'GPU not detected! Please go to Runtime -> Change runtime type -> T4 GPU.'
print(f'CUDA Device: {torch.cuda.get_device_name(0)}')
print(f'PyTorch Version: {torch.__version__}')
print(f'Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## Step 3: Install Required Dependencies
Installs MedMNIST, ReportLab, Matplotlib, PyTest, and Weights & Biases (`wandb`).

In [ ]:
!pip install --quiet medmnist reportlab matplotlib pytest wandb

## Step 4: Configure Weights & Biases via Colab Secrets
Reads the W&B API key securely from **Colab Secrets** (click the **key icon** on the left sidebar, add name: `WANDB_API_KEY`, value: your API key from [https://wandb.ai/authorize](https://wandb.ai/authorize)).
No pasted key variable is stored in the notebook.

In [ ]:
import os
import wandb

try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
    if wandb_key:
        os.environ['WANDB_API_KEY'] = wandb_key.strip()
        wandb.login(key=wandb_key.strip())
        wandb_mode = 'online'
        print('W&B online logging successfully initialized from Colab Secrets.')
    else:
        raise ValueError('WANDB_API_KEY secret is empty.')
except Exception as e:
    print(f'Notice: WANDB_API_KEY not found in Colab Secrets ({e}).')
    print('To log online: Add WANDB_API_KEY in Colab Secrets (key icon on left sidebar).')
    print('Setting wandb_mode = "offline". Metrics will be saved locally.')
    wandb_mode = 'offline'

## Step 5: Unpack Project Code
If running from `colab_package.zip`, extract it into the workspace.

In [ ]:
import os
if os.path.exists('colab_package.zip'):
    !unzip -q -o colab_package.zip
    print('Successfully extracted colab_package.zip')
else:
    print('Using repository directory directly.')
!ls -la

## Step 6: Generate Verified Partitions & Verify Committed Initial Checkpoints
Downloads official PathMNIST and builds verified Dirichlet partitions ($\alpha=0.3$) with 40% label noise on Client 0.
Verifies the pre-committed deterministic initial model checkpoints and manifest (zero on-the-fly model re-initialization).

In [ ]:
!python generate_pathmnist_data.py

# Verify committed deterministic initial checkpoints against manifest
import json, hashlib, os

manifest_path = "checkpoints/initial/manifest.json"
assert os.path.exists(manifest_path), f"Missing {manifest_path}!"
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

for seed in [42, 123, 2026]:
    ckpt_path = f"checkpoints/initial/initial_model_seed_{seed}.pt"
    assert os.path.exists(ckpt_path), f"Missing checkpoint: {ckpt_path}!"
    with open(ckpt_path, "rb") as f:
        sha = hashlib.sha256(f.read()).hexdigest()
    expected = manifest[str(seed)]["sha256"]
    assert sha == expected, f"Hash mismatch for seed {seed}! {sha} != {expected}"
    print(f"Verified seed {seed} initial model SHA256: {sha}")

print("All committed initial model checkpoints verified against manifest!")

## Step 7: Run Unit & Correctness Test Suite on GPU Runtime
Executes all 11 unit tests verifying partition integrity, model architecture, momentum reset, CUDA-tolerant $\mu=0$ equivalence, and resume reproducibility.

In [ ]:
!python -m pytest tests/ -v

## Step 8: Fast 5-Round Pilot Run
Runs 5 communication rounds across all 4 experimental groups on seed 42 to verify timing, GPU memory, and metric collection in ~2 minutes before launching the full matrix.

In [ ]:
!python -u orchestrator.py --device cuda --pilot --wandb_mode $wandb_mode --results_dir /content/drive/MyDrive/PathMNIST_Results/pilot

## Step 9: Launch Full 15-Run Experiment Matrix on GPU
Autonomous execution pipeline:
- **Phase 1:** $\mu$ sweep on noisy seed 42 ($\mu \in \{0.001, 0.01, 0.1\}$)
- **Phase 2:** $\mu$ selection using rounds 91–100 mean validation accuracy
- **Phase 3:** Full 2×2 factorial matrix across seeds 42, 123, 2026 (100 rounds each)
- **Phase 4:** Compilation of deliverables (tables, plots, and PDF)

All checkpoints, JSON histories, and artifacts persist directly to Google Drive.

In [ ]:
!python -u orchestrator.py --device cuda --wandb_mode $wandb_mode --results_dir /content/drive/MyDrive/PathMNIST_Results/final

## Step 10: (Optional) Sync Offline W&B Runs to Cloud Dashboard
If you ran the experiment with `wandb_mode='offline'`, run this cell to sync all logged runs to your online W&B project:

In [ ]:
!wandb sync --include-offline /content/drive/MyDrive/PathMNIST_Results/final/wandb/

## Step 11: Inspect Final Results & Summary Tables

In [ ]:
from IPython.display import Markdown, display
import os

results_md = '/content/drive/MyDrive/PathMNIST_Results/final/tables/results.md'
if os.path.exists(results_md):
    with open(results_md, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print(f'Table file not found yet at {results_md}')

## Step 12: View Key Diagnostic Plots Inline

In [ ]:
from IPython.display import Image, display
import os

plots_dir = '/content/drive/MyDrive/PathMNIST_Results/final/plots'
p1 = os.path.join(plots_dir, 'plot1_test_accuracy_vs_rounds.png')
p9 = os.path.join(plots_dir, 'plot9_aggregation_weights.png')
p_sweep = os.path.join(plots_dir, 'plot_mu_sweep.png')

if os.path.exists(p1):
    print('Test Accuracy Curves (Solid: Mean ± Std, Dotted: Per-Seed):')
    display(Image(p1))
if os.path.exists(p9):
    print('Client Aggregation Weights Across Seeds:')
    display(Image(p9))
if os.path.exists(p_sweep):
    print('Mu Sweep Validation Convergence:')
    display(Image(p_sweep))

## Step 13: Download All Deliverables to Local Machine

In [ ]:
!zip -r /content/PathMNIST_Final_Results.zip /content/drive/MyDrive/PathMNIST_Results/final/ README.md DEVIATIONS.md

from google.colab import files
if os.path.exists('/content/PathMNIST_Final_Results.zip'):
    files.download('/content/PathMNIST_Final_Results.zip')
pdf_path = '/content/drive/MyDrive/PathMNIST_Results/final/PathMNIST_FedAvg_vs_FedProx_Report.pdf'
if os.path.exists(pdf_path):
    files.download(pdf_path)